In [1]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sys

print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)
print("All good!")

Python location: c:\Users\chity\OneDrive\Desktop\Credit-risk-rag-project\credit_risk_env\Scripts\python.exe
Pandas version: 3.0.5
All good!


In [9]:
df_sample = pd.read_csv("data/raw/accepted_2007_to_2018Q4.csv", nrows=1000)
print("Shape:", df_sample.shape)
df_sample.head()

Shape: (1000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# How many columns do we have, and what are they?
print("Number of columns:", df_sample.shape[1])
print("\nColumn names:")
print(df_sample.columns.tolist())

Number of columns: 151

Column names:
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verific

In [1]:
import pandas as pd

chunk_size = 100000  # process 100,000 rows at a time
missing_counts = None
total_rows = 0

for chunk in pd.read_csv("data/raw/accepted_2007_to_2018Q4.csv", 
                           chunksize=chunk_size, low_memory=False):
    if missing_counts is None:
        missing_counts = chunk.isnull().sum()
    else:
        missing_counts += chunk.isnull().sum()
    total_rows += len(chunk)
    print(f"Processed {total_rows} rows so far...", end="\r")

print("\n\nTotal rows processed:", total_rows)

# Convert to percentage missing per column
missing_percent = (missing_counts / total_rows * 100).sort_values(ascending=False)
print(missing_percent)

Processed 2260701 rows so far...

Total rows processed: 2260701
member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_reason                                99.517097
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
                                                 ...    
total_rec_int                                   0.001460
disbursement_method                             0.001460
hardship_flag                                   0.001460
debt_settlement_flag                            0.001460
id                                              0.000000
Length: 151, dtype: float64


In [2]:
pd.set_option('display.max_rows', 200)  # let pandas show all 151 rows, not truncate
print(missing_percent)

member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_reason                                99.517097
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
payment_plan_start_date                        99.517097
hardship_type                                  99.517097
hardship_status                                99.517097
hardship_start_date                            99.517097
deferral_term                                  99.517097
hardship_amount                                99.517097
hardship_dpd                                   99.517097
hardship_loan_status                           99.517097
hardship_length                                99.517097
hardship_end_date                              99.517097
settlement_status                              98.485160
debt_settlement_flag_date                      98.485160
settlement_term                

In [3]:
# Columns we've decided to KEEP, based on our missing-value + data-leakage analysis
keep_columns = [
    # Target variable
    'loan_status',
    # Core application info
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_title', 'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'purpose', 'title', 'zip_code', 'addr_state', 'dti',
    'issue_d',
    # Credit history
    'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'delinq_2yrs',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'acc_now_delinq',
    # Additional useful, low-missing features
    'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'avg_cur_bal',
    'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'num_actv_bc_tl',
    'num_bc_sats', 'num_sats', 'acc_open_past_24mths', 'mort_acc',
    'total_bal_ex_mort', 'total_bc_limit', 'chargeoff_within_12_mths',
    'delinq_amnt', 'tax_liens', 'collections_12_mths_ex_med'
]
# Remove any accidental duplicates in our list
keep_columns = list(dict.fromkeys(keep_columns))

print("Total columns we're keeping:", len(keep_columns))
print(keep_columns)

Total columns we're keeping: 48
['loan_status', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'issue_d', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_rev_tl_op', 'num_actv_bc_tl', 'num_bc_sats', 'num_sats', 'acc_open_past_24mths', 'total_bal_ex_mort', 'total_bc_limit', 'chargeoff_within_12_mths', 'delinq_amnt', 'tax_liens', 'collections_12_mths_ex_med']


In [4]:
df = pd.read_csv(
    "data/raw/accepted_2007_to_2018Q4.csv",
    usecols=keep_columns,
    low_memory=False
)
print("Shape:", df.shape)
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Shape: (2260701, 48)
Memory usage (MB): 2312.19


In [6]:
df.to_csv("data/processed/loans_trimmed.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [7]:
df_check = pd.read_csv("data/processed/loans_trimmed.csv")
print("Reloaded shape:", df_check.shape)
print("Matches original?", df_check.shape == df.shape)

Reloaded shape: (2260701, 48)
Matches original? True


In [8]:
df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,delinq_amnt,mo_sin_old_rev_tl_op,mort_acc,num_actv_bc_tl,num_bc_sats,num_sats,pub_rec_bankruptcies,tax_liens,total_bal_ex_mort,total_bc_limit
0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,...,0.0,128.0,1.0,2.0,2.0,7.0,0.0,0.0,7746.0,2400.0
1,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,...,0.0,192.0,4.0,5.0,13.0,22.0,0.0,0.0,39475.0,79300.0
2,20000.0,60 months,10.78,432.66,B,B4,truck driver,10+ years,MORTGAGE,63000.0,...,0.0,184.0,5.0,2.0,2.0,6.0,0.0,0.0,18696.0,6200.0
3,35000.0,60 months,14.85,829.90,C,C5,Information Systems Officer,10+ years,MORTGAGE,110000.0,...,0.0,87.0,1.0,4.0,8.0,13.0,0.0,0.0,52226.0,62500.0
4,10400.0,60 months,22.45,289.91,F,F1,Contract Specialist,3 years,MORTGAGE,104433.0,...,0.0,210.0,6.0,4.0,5.0,12.0,0.0,0.0,95768.0,20300.0


In [9]:
df.dtypes

loan_amnt                     float64
term                              str
int_rate                      float64
installment                   float64
grade                             str
sub_grade                         str
emp_title                         str
emp_length                        str
home_ownership                    str
annual_inc                    float64
verification_status               str
issue_d                           str
loan_status                       str
purpose                           str
title                             str
zip_code                          str
addr_state                        str
dti                           float64
delinq_2yrs                   float64
earliest_cr_line                  str
fico_range_low                float64
fico_range_high               float64
inq_last_6mths                float64
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util  

In [10]:
df['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [11]:
# Step 1: Define which statuses count as "Paid" (0) vs "Default" (1)

paid_statuses = [
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid'
]
default_statuses = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
]
# Step 2: Keep only rows with a FINAL, known outcome (drop Current, Late, In Grace Period)
df_clean = df[df['loan_status'].isin(paid_statuses + default_statuses)].copy()
print("Rows before filtering:", len(df))
print("Rows after filtering:", len(df_clean))
# Step 3: Create the binary target column
df_clean['default'] = df_clean['loan_status'].apply(
    lambda status: 1 if status in default_statuses else 0
)
# Step 4: Check the result
print("\nTarget variable distribution:")
print(df_clean['default'].value_counts())
print("\nAs percentages:")
print(df_clean['default'].value_counts(normalize=True) * 100)

Rows before filtering: 2260701
Rows after filtering: 1348099

Target variable distribution:
default
0    1078739
1     269360
Name: count, dtype: int64

As percentages:
default
0    80.019272
1    19.980728
Name: proportion, dtype: float64


In [12]:
df_clean.to_csv("data/processed/loans_with_target.csv", index=False)
print("Saved!")

Saved!
